# QP Forward Example

This notebook loads a structured QP from `notebooks/data/qp`, defines the model with the public `NLPOptNet` API, trains it, then reloads the saved metadata for prediction.
        

In [ ]:
from pathlib import Path
import sys

import numpy as np

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / 'nlpoptnet').is_dir() and (path / 'notebooks').is_dir()
)
SRC = ROOT / 'nlpoptnet' / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from nlpoptnet import NLPOptNet

In [ ]:
CONFIG = {
    'epochs': 3,
    'batch_size': 8,
    'learning_rate': 1e-3,
    'train_frac': 0.8,
    'hidden_size': 32,
    'hidden_layers': 2,
    'seed': 42,
    'dtype': 'float64',
    'print_every': 1,
    'verbose': True,
}

DATA_DIR = ROOT / 'notebooks' / 'data' / 'qp'

## Build the structured QP

The matrices are read from `problem.npz` and become attributes such as `model.Q`, `model.A`, and `model.l`.
        

In [ ]:
model = NLPOptNet(config=CONFIG, type='qp', name='notebook_qp')
x = model.add_parameter(['x1', 'x2'])
y = model.add_variable(['y1', 'y2', 'y3', 'y4'])

model.extract(DATA_DIR / 'problem.npz')
model.objective(0.5 * model.quad(model.Q, y) + model.lin(model.c, y))
model.constraints.equality.add(
    model.lin(model.A, y) == model.b + model.lin(model.B, x),
)
model.constraints.inequality.add(
    model.lin(model.C, y) <= model.d + model.lin(model.D, x),
)
model.constraints.box.add(
    var=y,
    lower=model.l + model.lin(model.L, x),
    upper=model.u + model.lin(model.U, x),
)
model.dataset(parameters=DATA_DIR / 'parameters.csv')
model.build()
result = model.optimize()
run_dir = Path(result['output_dir'])
run_dir
        

## Reload and predict
        

In [ ]:
sample_x = np.loadtxt(DATA_DIR / 'parameters.csv', delimiter=',', max_rows=1)
reloaded = NLPOptNet().load(run_dir / 'metadata.json')
reloaded.predict(sample_x)
        